# Modded-NanoGPT

This notebook trains a *NanoGPT* model to use 8 NVIDIA H100 GPUs to attains 3.28 cross-entropy loss on the [FineWeb](https://huggingface.co/datasets/HuggingFaceFW/fineweb) validation set.

The target (3.28 validation loss on FineWeb) follows Andrej Karpathy's [GPT-2 replication in llm.c, which attains that loss after running for 45 minutes](https://github.com/karpathy/llm.c/discussions/481#:~:text=By%20the%20end%20of%20the%20optimization%20we%27ll%20get%20to%20about%203.29).

In [ ]:
# Setup fast transfer libraries
%env HF_TRANSFER=1
%env HF_HUB_ENABLE_HF_TRANSFER=1

env: HF_TRANSFER=1
env: HF_HUB_ENABLE_HF_TRANSFER=1
env: MODDED_NANOGPT_CACHE=/home/ubuntu/workspace/modded-nanogpt/data/


In [ ]:
import os

DATASET_PATH = os.path.join(os.getcwd(), "data")
os.makedirs(DATASET_PATH, exist_ok=True)

os.environ["DATASET_PATH"] = DATASET_PATH

In [ ]:
# download fine web
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

from huggingface_hub import hf_hub_download


def get(fname):
    local_dir = os.path.join("data", "fineweb10B")
    if not os.path.exists(os.path.join(local_dir, fname)):
        hf_hub_download(
            repo_id="kjj0/fineweb10B-gpt2",
            filename=fname,
            repo_type="dataset",
            local_dir=local_dir,
        )


num_chunks = 8  # full fineweb10B use 103. Each chunk is 100M tokens

files = ["fineweb_val_%06d.bin" % 0] + [
    "fineweb_train_%06d.bin" % i for i in range(1, num_chunks + 1)
]

with ThreadPoolExecutor(max_workers=8) as executor:  # adjust workers
    futures = {executor.submit(get, f): f for f in files}
    for future in as_completed(futures):
        fname = futures[future]
        try:
            future.result()
            print(f"Downloaded {fname}")
        except Exception as e:
            print(f"Failed {fname}: {e}")

Downloaded fineweb_train_000001.bin
Downloaded fineweb_train_000002.bin
Downloaded fineweb_val_000000.bin
Downloaded fineweb_train_000005.bin
Downloaded fineweb_train_000004.bin
Downloaded fineweb_train_000003.bin
Downloaded fineweb_train_000006.bin
Downloaded fineweb_train_000007.bin
Downloaded fineweb_train_000008.bin


In [15]:
# print all the version for the following libraries: pytorc, cuda
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name()}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"SMs: {props.multi_processor_count}")
else:
    print("WARNING NO VALID CUDA SETUP FOUND")

PyTorch version: 2.8.0+cu128
CUDA version: 12.8
CUDA available: True
CUDA device count: 8
Current CUDA device: 0
CUDA device name: NVIDIA H100 80GB HBM3
GPU: NVIDIA H100 80GB HBM3
SMs: 132
